In [1]:
"""多智能体旅行规划系统"""

import json
from typing import Dict, Any, List
# jlq_rpc # =====================================
# from hello_agents import SimpleAgent
# from ...services.llm_service import get_llm

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import InMemorySaver


# jlq_crt # =====================================
# from hello_agents.tools import MCPTool
from tools.builtin.protocol_tools import MCPTool

from backend.app.models.schemas import TripRequest, TripPlan, DayPlan, Attraction, Meal, WeatherInfo, Location, Hotel
from backend.app.config import get_settings




E:\XXXX_CodeTool\anaconda\envs\torch_py314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\XXXX_CodeTool\anaconda\envs\torch_py314\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


应用名称: 正在建设alpha_0.0
版本: 0.0.1
服务器: 0.0.0.0:8000
高德地图API Key: 已配置
LLM API Key: 已配置
LLM Base URL: https://ws-3pjqp2yl3m99h27j.cn-beijing.maas.aliyuncs.com/compatible-mode/v1
LLM Model: glm-5.2
日志级别: INFO


In [2]:
# ============ Agent提示词 ============

ATTRACTION_AGENT_PROMPT = """你是景点搜索专家。你的任务是根据城市和用户偏好搜索合适的景点。

**重要提示:**
你必须使用工具来搜索景点!不要自己编造景点信息!

**工具调用格式:**
使用maps_text_search工具时,必须严格按照以下格式:
`[TOOL_CALL:amap_maps_text_search:keywords=景点关键词,city=城市名]`

**示例:**
用户: "搜索北京的历史文化景点"
你的回复: [TOOL_CALL:amap_maps_text_search:keywords=历史文化,city=北京]

用户: "搜索上海的公园"
你的回复: [TOOL_CALL:amap_maps_text_search:keywords=公园,city=上海]

**注意:**
1. 必须使用工具,不要直接回答
2. 格式必须完全正确,包括方括号和冒号
3. 参数用逗号分隔
"""

WEATHER_AGENT_PROMPT = """你是天气查询专家。你的任务是查询指定城市的天气信息。

**重要提示:**
你必须使用工具来查询天气!不要自己编造天气信息!

**工具调用格式:**
使用maps_weather工具时,必须严格按照以下格式:
`[TOOL_CALL:amap_maps_weather:city=城市名]`

**示例:**
用户: "查询北京天气"
你的回复: [TOOL_CALL:amap_maps_weather:city=北京]

用户: "上海的天气怎么样"
你的回复: [TOOL_CALL:amap_maps_weather:city=上海]

**注意:**
1. 必须使用工具,不要直接回答
2. 格式必须完全正确,包括方括号和冒号
"""

HOTEL_AGENT_PROMPT = """你是酒店推荐专家。你的任务是根据城市和景点位置推荐合适的酒店。

**重要提示:**
你必须使用工具来搜索酒店!不要自己编造酒店信息!

**工具调用格式:**
使用maps_text_search工具搜索酒店时,必须严格按照以下格式:
`[TOOL_CALL:amap_maps_text_search:keywords=酒店,city=城市名]`

**示例:**
用户: "搜索北京的酒店"
你的回复: [TOOL_CALL:amap_maps_text_search:keywords=酒店,city=北京]

**注意:**
1. 必须使用工具,不要直接回答
2. 格式必须完全正确,包括方括号和冒号
3. 关键词使用"酒店"或"宾馆"
"""

PLANNER_AGENT_PROMPT = """你是行程规划专家。你的任务是根据景点信息和天气信息,生成详细的旅行计划。

请严格按照以下JSON格式返回旅行计划:
```json
{
  "city": "城市名称",
  "start_date": "YYYY-MM-DD",
  "end_date": "YYYY-MM-DD",
  "days": [
    {
      "date": "YYYY-MM-DD",
      "day_index": 0,
      "description": "第1天行程概述",
      "transportation": "交通方式",
      "accommodation": "住宿类型",
      "hotel": {
        "name": "酒店名称",
        "address": "酒店地址",
        "location": {"longitude": 116.397128, "latitude": 39.916527},
        "price_range": "300-500元",
        "rating": "4.5",
        "distance": "距离景点2公里",
        "type": "经济型酒店",
        "estimated_cost": 400
      },
      "attractions": [
        {
          "name": "景点名称",
          "address": "详细地址",
          "location": {"longitude": 116.397128, "latitude": 39.916527},
          "visit_duration": 120,
          "description": "景点详细描述",
          "category": "景点类别",
          "ticket_price": 60
        }
      ],
      "meals": [
        {"type": "breakfast", "name": "早餐推荐", "description": "早餐描述", "estimated_cost": 30},
        {"type": "lunch", "name": "午餐推荐", "description": "午餐描述", "estimated_cost": 50},
        {"type": "dinner", "name": "晚餐推荐", "description": "晚餐描述", "estimated_cost": 80}
      ]
    }
  ],
  "weather_info": [
    {
      "date": "YYYY-MM-DD",
      "day_weather": "晴",
      "night_weather": "多云",
      "day_temp": 25,
      "night_temp": 15,
      "wind_direction": "南风",
      "wind_power": "1-3级"
    }
  ],
  "overall_suggestions": "总体建议",
  "budget": {
    "total_attractions": 180,
    "total_hotels": 1200,
    "total_meals": 480,
    "total_transportation": 200,
    "total": 2060
  }
}
```

**重要提示:**
1. weather_info数组必须包含每一天的天气信息
2. 温度必须是纯数字(不要带°C等单位)
3. 每天安排2-3个景点
4. 考虑景点之间的距离和游览时间
5. 每天必须包含早中晚三餐
6. 提供实用的旅行建议
7. **必须包含预算信息**:
   - 景点门票价格(ticket_price)
   - 餐饮预估费用(estimated_cost)
   - 酒店预估费用(estimated_cost)
   - 预算汇总(budget)包含各项总费用
"""

In [3]:
class MultiAgentTripPlanner:
    """多智能体旅行规划系统"""

    def __init__(self,llm,checkpointer):
        """初始化多智能体系统"""
        print("🔄 开始初始化多智能体旅行规划系统...")

        try:
            self.settings = get_settings()
            self.llm = llm
            self.checkpointer = checkpointer

            # 创建共享的MCP工具(只创建一次)


            # 创建景点搜索Agent
            # print("  - 创建景点搜索Agent...")
            # self.attraction_agent = SimpleAgent(
            #     name="景点搜索专家",
            #     llm=self.llm,
            #     system_prompt=ATTRACTION_AGENT_PROMPT
            # )
            # self.attraction_agent = create_agent(
            #         model=self.llm,
            #         tools=tools,
            #         system_prompt=ATTRACTION_AGENT_PROMPT,
            #         checkpointer=self.checkpointer,
            #         )
            #
            # self.attraction_agent.add_tool(tools)

            # # 创建天气查询Agent
            # print("  - 创建天气查询Agent...")
            # self.weather_agent = SimpleAgent(
            #     name="天气查询专家",
            #     llm=self.llm,
            #     system_prompt=WEATHER_AGENT_PROMPT
            # )
            # self.weather_agent.add_tool(self.amap_tool)
            #
            # # 创建酒店推荐Agent
            # print("  - 创建酒店推荐Agent...")
            # self.hotel_agent = SimpleAgent(
            #     name="酒店推荐专家",
            #     llm=self.llm,
            #     system_prompt=HOTEL_AGENT_PROMPT
            # )
            # self.hotel_agent.add_tool(self.amap_tool)
            #
            # # 创建行程规划Agent(不需要工具)
            # print("  - 创建行程规划Agent...")
            # self.planner_agent = SimpleAgent(
            #     name="行程规划专家",
            #     llm=self.llm,
            #     system_prompt=PLANNER_AGENT_PROMPT
            # )

            # print(f"✅ 多智能体系统初始化成功")
            # print(f"   景点搜索Agent: {len(tools)} 个工具")  # ✅ 直接用 tools
            # for t in tools:
                # print(f"     🔧 {t.name}: {t.description[:50]}")
            # print(f"   景点搜索Agent: {len(self.attraction_agent.list_tools())} 个工具")
            # print(f"   天气查询Agent: {len(self.weather_agent.list_tools())} 个工具")
            # print(f"   酒店推荐Agent: {len(self.hotel_agent.list_tools())} 个工具")

        except Exception as e:
            print(f"❌ 多智能体系统初始化失败: {str(e)}")
            import traceback
            traceback.print_exc()
            raise

    async def Agentinit(self):
        print("  - 正在创建共享MCP工具...")
        self.mcp_client = MultiServerMCPClient(
                    {
                        "amap": {
                                    "url": "https://mcp.amap.com/sse?key=f5345c390cff8a23d5c42a395b765779",
                                    "transport": "sse",
                                }, # 记得后续把map的key隐藏一下。
                    }
                )
        tools = await self.mcp_client.get_tools()

         # 创建景点搜索Agent
        print("  - 创建景点搜索Agent...")
        self.attraction_agent = create_agent(
                    model=self.llm,
                    tools=tools,
                    system_prompt=ATTRACTION_AGENT_PROMPT,
                    checkpointer=self.checkpointer,
                    )

        print(f"✅ 多智能体系统初始化成功")
        print(f"   景点搜索Agent: {len(tools)} 个工具")  # ✅ 直接用 tools
        for t in tools:
            print(f"     🔧 {t.name}: {t.description[:50]}")


    # ↑ ↑ ↑ 以上功能测试完好，正在进行下一步测试。=====================================
    def plan_trip(self, request: TripRequest) -> TripPlan:
        """
        使用多智能体协作生成旅行计划

        Args:
            request: 旅行请求

        Returns:
            旅行计划
        """
        try:
            print(f"\n{'='*60}")
            print(f"🚀 开始多智能体协作规划旅行...")
            print(f"目的地: {request.city}")
            print(f"日期: {request.start_date} 至 {request.end_date}")
            print(f"天数: {request.travel_days}天")
            print(f"偏好: {', '.join(request.preferences) if request.preferences else '无'}")
            print(f"{'='*60}\n")

            # 步骤1: 景点搜索Agent搜索景点
            print("📍 步骤1: 搜索景点...")
            attraction_query = self._build_attraction_query(request)
            attraction_response = self.attraction_agent.run(attraction_query)
            print(f"景点搜索结果: {attraction_response[:200]}...\n")

            # 步骤2: 天气查询Agent查询天气
            print("🌤️  步骤2: 查询天气...")
            weather_query = f"请查询{request.city}的天气信息"
            weather_response = self.weather_agent.run(weather_query)
            print(f"天气查询结果: {weather_response[:200]}...\n")

            # 步骤3: 酒店推荐Agent搜索酒店
            print("🏨 步骤3: 搜索酒店...")
            hotel_query = f"请搜索{request.city}的{request.accommodation}酒店"
            hotel_response = self.hotel_agent.run(hotel_query)
            print(f"酒店搜索结果: {hotel_response[:200]}...\n")

            # 步骤4: 行程规划Agent整合信息生成计划
            print("📋 步骤4: 生成行程计划...")
            planner_query = self._build_planner_query(request, attraction_response, weather_response, hotel_response)
            planner_response = self.planner_agent.run(planner_query)
            print(f"行程规划结果: {planner_response[:300]}...\n")

            # 解析最终计划
            trip_plan = self._parse_response(planner_response, request)

            print(f"{'='*60}")
            print(f"✅ 旅行计划生成完成!")
            print(f"{'='*60}\n")

            return trip_plan

        except Exception as e:
            print(f"❌ 生成旅行计划失败: {str(e)}")
            import traceback
            traceback.print_exc()
            return self._create_fallback_plan(request)



    def _build_attraction_query(self, request: TripRequest) -> str:
        """构建景点搜索查询 - 直接包含工具调用"""
        keywords = []
        if request.preferences:
            # 只取第一个偏好作为关键词
            keywords = request.preferences[0]
        else:
            keywords = "景点"

        # 直接返回工具调用格式
        query = f"请使用amap_maps_text_search工具搜索{request.city}的{keywords}相关景点。\n[TOOL_CALL:amap_maps_text_search:keywords={keywords},city={request.city}]"
        return query

        def _build_planner_query(self, request: TripRequest, attractions: str, weather: str, hotels: str = "") -> str:
        """构建行程规划查询"""
        query = f"""请根据以下信息生成{request.city}的{request.travel_days}天旅行计划:

**基本信息:**
- 城市: {request.city}
- 日期: {request.start_date} 至 {request.end_date}
- 天数: {request.travel_days}天
- 交通方式: {request.transportation}
- 住宿: {request.accommodation}
- 偏好: {', '.join(request.preferences) if request.preferences else '无'}

**景点信息:**
{attractions}

**天气信息:**
{weather}

**酒店信息:**
{hotels}

**要求:**
1. 每天安排2-3个景点
2. 每天必须包含早中晚三餐
3. 每天推荐一个具体的酒店(从酒店信息中选择)
3. 考虑景点之间的距离和交通方式
4. 返回完整的JSON格式数据
5. 景点的经纬度坐标要真实准确
"""
        if request.free_text_input:
            query += f"\n**额外要求:** {request.free_text_input}"

        return query

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()

# API URL
llm_api_url = os.getenv("LLM_BASE_URL")
print(llm_api_url)

https://ws-3pjqp2yl3m99h27j.cn-beijing.maas.aliyuncs.com/compatible-mode/v1


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()


async def main():
    llm = ChatOpenAI(
                        # model="gemma4:latest",
                        # base_url="http://localhost:11434/v1",   # Ollama 的 OpenAI 兼容端点
                        # api_key="ollama",                        # 随便填，Ollama 不验证
                        model=os.getenv("LLM_MODEL_ID"),
                        base_url=os.getenv("LLM_BASE_URL"),
                        api_key=os.getenv("LLM_API_KEY"),
                        temperature=0.7,
    )

    # llm = ChatZhipuAI(
    #     model="glm-5.2",
    #     temperature=0.5,
    # )

    checkpointer = InMemorySaver()

    planner = MultiAgentTripPlanner(llm =llm,checkpointer=checkpointer)
    await planner.Agentinit()

    # 正常使用
    result = await planner.attraction_agent.ainvoke(
        {"messages": [{"role": "user", "content": "推荐北京的景点"}]},
        config={"configurable": {"thread_id": "1"}},
    )
    print("agent执行结果：",result["messages"][-1].content)

    # 清理
    # await planner.close()


if __name__ == "__main__":
    # ✅ 方案 A：直接 await（Jupyter 支持顶层 await）
    await main()

🔄 开始初始化多智能体旅行规划系统...
  - 正在创建共享MCP工具...
  - 创建景点搜索Agent...
✅ 多智能体系统初始化成功
   景点搜索Agent: 15 个工具
     🔧 maps_direction_bicycling: 骑行路径规划用于规划骑行通勤方案，规划时会考虑天桥、单行线、封路等情况。最大支持 500km 的骑行
     🔧 maps_direction_driving: 驾车路径规划 API 可以根据用户起终点经纬度坐标规划以小客车、轿车通勤出行的方案，并且返回通勤方案
     🔧 maps_direction_transit_integrated: 根据用户起终点经纬度坐标规划综合各类公共（火车、公交、地铁）交通方式的通勤方案，并且返回通勤方案的数
     🔧 maps_direction_walking: 根据输入起点终点经纬度坐标规划100km 以内的步行通勤方案，并且返回通勤方案的数据
     🔧 maps_distance: 测量两个经纬度坐标之间的距离,支持驾车、步行以及球面距离测量
     🔧 maps_geo: 将详细的结构化地址转换为经纬度坐标。支持对地标性名胜景区、建筑物名称解析为经纬度坐标
     🔧 maps_regeocode: 将一个高德经纬度坐标转换为行政区划地址信息
     🔧 maps_ip_location: IP 定位根据用户输入的 IP 地址，定位 IP 的所在位置
     🔧 maps_schema_personal_map: 用于行程规划结果在高德地图展示。将行程规划位置点按照行程顺序填入lineList，返回结果为高德地图
     🔧 maps_around_search: 周边搜，根据用户传入关键词以及坐标location，搜索出radius半径范围的POI
     🔧 maps_search_detail: 查询关键词搜或者周边搜获取到的POI ID的详细信息
     🔧 maps_text_search: 关键字搜索 API 根据用户输入的关键字进行 POI 搜索，并返回相关的信息
     🔧 maps_schema_navi:  Schema唤醒客户端-导航页面，用于根据用户输入终点信息，返回一个拼装好的客户端唤醒URI，用户
     🔧 map